###**Into the SpiderVerse !**
Classify Spider People Using ResNet50<br><br>
Before you begin:
- Ensure that you have selected T4 GPU Runtime
- You have uploaded the aug_dataset_with_unknown.zip folder to Google Drive
<br>
Let's begin !!!

### Step 1: Mount Google Drive
We do this to access the dataset stored in our Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Extract the Uploaded Dataset
The dataset is in the form of a ZIP file. We will extract it into a folder (Unzip) it to access the images inside

In [2]:
import zipfile

zip_path = "/content/drive/MyDrive/ML_Datasets/aug_dataset_with_unknown.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Dataset extracted successfully!")

Dataset extracted successfully!


In [3]:
import os

dataset_path = "/content/dataset"

for root, dirs, files in os.walk(dataset_path):
    print(root, "->", len(files), "images")

/content/dataset -> 0 images
/content/dataset/__MACOSX -> 1 images
/content/dataset/__MACOSX/aug_dataset_with_unknown -> 10 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/hobie_brown -> 394 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/spider_ham -> 398 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/miles_morales -> 395 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/pavitr_prabhakar -> 394 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/miguel_ohara -> 395 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/spider_man_noir -> 396 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/not_spider_person -> 396 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/gwen_stacy -> 397 images
/content/dataset/__MACOSX/aug_dataset_with_unknown/peter_b_parker -> 393 images
/content/dataset/aug_dataset_with_unknown -> 1 images
/content/dataset/aug_dataset_with_unknown/hobie_brown -> 394 images
/content/dataset/aug_dataset_with_u

### Step 3: Split Dataset into Train and Test Subsets
This will allow us to train and check the accuracy of the model we trained

In [18]:
import random
import shutil

source_dir = "/content/dataset/aug_dataset_with_unknown"
train_dir = "/content/dataset/train"
val_dir = "/content/dataset/val"
test_dir="/content/dataset/test"

split_ratio = 0.8

classes = os.listdir(source_dir)

for cls in classes:
    if cls == ".DS_Store":
        continue
    class_path = os.path.join(source_dir, cls)
    images = os.listdir(class_path)

    random.shuffle(images)

    split_index = int(len(images) * split_ratio)

    train_images = images[:split_index]
    remaining_images = images[split_index:]

    # Split remaining images into val and test
    half = len(remaining_images) // 2
    val_images = remaining_images[:half]
    test_images = remaining_images[half:]


    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(val_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)

    for img in train_images:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(train_dir, cls, img)
        )

    for img in val_images:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(val_dir, cls, img)
        )
    for img in test_images:
        shutil.copy(
            os.path.join(class_path, img),
            os.path.join(test_dir, cls, img)
        )

print("Dataset split completed!")

Dataset split completed!


### Step 4: Check the Train and Test parts that we have split
Check how many images are there for each Spider-person

In [19]:

for root, dirs, files in os.walk("/content/dataset/test"):
    print(root, len(files))

/content/dataset/test 0
/content/dataset/test/hobie_brown 40
/content/dataset/test/spider_ham 40
/content/dataset/test/miles_morales 40
/content/dataset/test/pavitr_prabhakar 40
/content/dataset/test/miguel_ohara 40
/content/dataset/test/spider_man_noir 40
/content/dataset/test/not_spider_person 40
/content/dataset/test/gwen_stacy 40
/content/dataset/test/peter_b_parker 40


### Step 5: Install the  library
This is the library that contains the function to invoke our ResNet50

In [6]:
pip install tensorflow

###Summary

Uses Transfer Learning

Base model: ResNet50

Input size: 224×224

Output classes: 9

Optimizer: Adam

Loss: Categorical Crossentropy

In [8]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = 224
BATCH_SIZE = 32

#rescaling just a bit
train_gen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(
    "dataset/train",
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val = val_gen.flow_from_directory(
    "dataset/val",
    target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dense(256,activation="relu")(x)
output = Dense(9,activation="softmax")(x)

model = Model(inputs=base_model.input,outputs=output)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train,
    validation_data=val,
    epochs=10
)

Found 2843 images belonging to 9 classes.
Found 715 images belonging to 9 classes.
Epoch 1/10


KeyboardInterrupt: 

###Updated CNN with
-The training dataset uses **data augmentation** to artificially increase dataset diversity.Prevents overfitting& Improves generalization<br>
-A Dropout layer (0.5) is added after the dense layer.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 9
EPOCHS = 20


# --- Data Generators ---
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest'
)

val_gen = ImageDataGenerator(rescale=1./255)

train = train_gen.flow_from_directory(
    "dataset/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val = val_gen.flow_from_directory(
    "dataset/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# --- Base model ---
base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # freeze base

# --- Custom top layers ---
x = base_model.output
x = Flatten()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)  # reduce overfitting
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# --- Compile model ---
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# --- Callbacks ---
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
rlr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

# --- Train model ---
history = model.fit(
    train,
    validation_data=val,
    epochs=EPOCHS,
    callbacks=[es, rlr]
)

# --- Optional: Fine-tune top layers ---
# Unfreeze last 10 layers of ResNet50
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False

# Compile with smaller LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Continue training for fine-tuning
history_fine = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=[es, rlr]
)

###Resnet with Improved Accuracy<br>
- ResNet specific normalization by using preprocess_input <br>
- Additional augmentation techniques were introduced<br>
- GlobalAveragePooling2D() instead of Flatten()-Reduces the number of parameters<br>
- The dense layer was increased from 256 → 512 neurons.<br>
- Last 30 layers of ResNet50 are unfrozen<br>
- Used ReducedOnPlateau and EarlyStopping

In [20]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# -------------------- Settings --------------------
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 9

# -------------------- Data Generators --------------------
#Data Augmentation
train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2],
    channel_shift_range=20,
    fill_mode="nearest"
)

val_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

#Load Dataset
train = train_gen.flow_from_directory(
    "dataset/train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val = val_gen.flow_from_directory(
    "dataset/val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

# -------------------- Base Model --------------------
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Freeze base model initially
base_model.trainable = False

# -------------------- Custom Classification Head --------------------
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# -------------------- Compile Model --------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# -------------------- Callbacks --------------------
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

# -------------------- Train Top Layers --------------------
history = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

# -------------------- Fine-Tuning --------------------
base_model.trainable = True

# Freeze most layers and train last 30 layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train,
    validation_data=val,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

# -------------------- Save Model --------------------
model.save("spiderverse_resnet50_model.keras")

Found 3426 images belonging to 9 classes.
Found 1005 images belonging to 9 classes.
Epoch 1/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 82s 657ms/step - accuracy: 0.6401 - loss: 1.1472 - val_accuracy: 0.8627 - val_loss: 0.4166 - learning_rate: 0.0010
Epoch 2/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 58s 534ms/step - accuracy: 0.8222 - loss: 0.5095 - val_accuracy: 0.9294 - val_loss: 0.2219 - learning_rate: 0.0010
Epoch 3/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 60s 556ms/step - accuracy: 0.8786 - loss: 0.3941 - val_accuracy: 0.9612 - val_loss: 0.1417 - learning_rate: 0.0010
Epoch 4/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 58s 535ms/step - accuracy: 0.8920 - loss: 0.3161 - val_accuracy: 0.9662 - val_loss: 0.1044 - learning_rate: 0.0010
Epoch 5/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 59s 542ms/step - accuracy: 0.9174 - loss: 0.2458 - val_accuracy: 0.9791 - val_loss: 0.0830 - learning_rate: 0.0010
Epoch 6/10
108/108 ━━━━━━━━━━━━━━━━━━━━ 59s 547ms/step - accuracy: 0.9273 - loss: 0.2263 - val_accuracy: 0.9891 - val_loss: 0.0546 - learning

In [21]:
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

test = test_gen.flow_from_directory(
    "/content/dataset/test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

loss, acc = model.evaluate(test)

print("Test Accuracy:", acc)

Found 360 images belonging to 9 classes.
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 359ms/step - accuracy: 0.9972 - loss: 0.0069
Test Accuracy: 0.9972222447395325


| Character     | Unique Features    |
| ------------- | ------------------ |
| Miles Morales | black + red suit   |
| Gwen Stacy    | white hood         |
| Miguel O'Hara | glowing blue/red   |
| Spider Ham    | cartoon pig        |
| Spider Noir   | black/white        |
| Spider Punk   | collage punk style |
